Test to check if the path generation agent follows it's prompt correctly

## Setup and Imports

In [1]:
import sys
from pathlib import Path

# Add parent directory to path to import modules
sys.path.append(str(Path().parent.absolute()))

from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModel
import networkx as nx
import asyncio
import json

In [2]:
# Import from HypothesisGeneration
from HypothesisGeneration import (
    load_graph_with_text_as_JSON,
    return_giant_component_of_graph,
    load_embeddings,
    generate_node_embeddings,
)
from HypothesisGeneration.agents import create_path_generation_agent
from HypothesisGeneration.utils import generate_path, create_path
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage

## Load Test Graph and Embeddings

In [3]:
# Setup paths
data_dir_source = Path("../inputs/initial_graph_biomed")

# Load graph
graph_name = "sparse_global_graph.graphml"
G = load_graph_with_text_as_JSON(data_dir=data_dir_source, graph_name=graph_name)
G = return_giant_component_of_graph(G)
G = nx.Graph(G)

print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

100%|██████████| 37951/37951 [00:00<00:00, 203848.86it/s]


Graph loaded with 24729 nodes and 30923 edges


In [4]:
# Load embedding model and tokenizer
tokenizer_model = "BAAI/bge-large-en-v1.5"
embedding_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
embedding_model = AutoModel.from_pretrained(tokenizer_model)

# Load or generate embeddings
embedding_name = "embeddings-sparse_global_graph-bge_large_en_v1.5.pkl"
try:
    node_embeddings = load_embeddings(f"{data_dir_source}/{embedding_name}")
    print("✅ Node embeddings loaded successfully")
except:
    print("⚠️ Node embeddings not found, generating new ones...")
    node_embeddings = generate_node_embeddings(G, embedding_tokenizer, embedding_model)
    print("✅ Node embeddings generated")

✅ Node embeddings loaded successfully


In [5]:
import functools

bound_generate_path = functools.partial(generate_path, G, embedding_tokenizer, embedding_model, node_embeddings)

## Create Tool and Agent

In [6]:
# Create the Path Generation Agent
path_agent = create_path_generation_agent(tools=[bound_generate_path])

print("🤖 Path Generation Agent created")
print(f"Agent name: {path_agent.name}")

🤖 Path Generation Agent created
Agent name: path_generation_agent


# Real call

In [7]:
path_list_for_vis, path_list_for_vis_string = create_path(
    G,
    embedding_tokenizer,
    embedding_model,
    node_embeddings,
    generate_graph_expansion=None,
    randomness_factor=0,
    num_random_waypoints=0,
    shortest_path=True,
    second_hop=False,
    data_dir="./",
    save_files=False,
    verbatim=False,
    keyword_1="diabetes",
    keyword_2="age",
)

In [8]:
print(path_list_for_vis_string)

diabetes||Disease -- is associated with; may share -- bladder cancer||Disease -- is a risk factor which increases the chance of developing -- age||Factor


# Agent call

In [9]:
test_message = "I want to explore correlations between diabetes and age."
print(f"📝 Test message: {test_message}")
input_message = TextMessage(content=test_message, source="user")

📝 Test message: I want to explore correlations between diabetes and age.


In [10]:
# Clear tool calls log
# Get agent response
print("🚀 Sending message to agent...")
response = await path_agent.run(task=input_message)
print(response.messages[-1].content)

🚀 Sending message to agent...
diabetes||Disease -- is associated with; may share -- bladder cancer||Disease -- is a risk factor which increases the chance of developing -- age||Factor
